# Pneumonia Detection from Chest X-rays
Fine-tune a YOLO classification model (transfer learning), optimize for **recall**, and explain predictions with **Grad-CAM**.

**Runtime:** Colab, T4 GPU. Add your Kaggle token as a Colab secret named `KAGGLE_API_TOKEN` (Notebook access on).

In [ ]:
import os
from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
!pip install -q --upgrade kagglehub kaggle ultralytics grad-cam scikit-learn

In [ ]:
import kagglehub
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
print(path)

## Train / val / test split
The dataset's own `val` folder has only 16 images, so train+val are merged and re-split 85/15. The original test set is untouched.

In [ ]:
import os, random
from pathlib import Path
from sklearn.model_selection import train_test_split

root = Path(path) / "chest_xray"
if not root.exists():
    root = next(Path(path).rglob("chest_xray"))

out = Path("/content/cxr")
random.seed(42)

def link(files, split, cls):
    d = out / split / cls
    d.mkdir(parents=True, exist_ok=True)
    for f in files:
        dst = d / f"{f.parent.parent.name}_{f.name}"
        if not dst.exists():
            os.symlink(f, dst)

for cls in ["NORMAL", "PNEUMONIA"]:
    files = list((root/"train"/cls).glob("*.jp*g")) + list((root/"val"/cls).glob("*.jp*g"))
    tr, va = train_test_split(files, test_size=0.15, random_state=42)
    link(tr, "train", cls)
    link(va, "val", cls)
    link(list((root/"test"/cls).glob("*.jp*g")), "test", cls)

for s in ["train", "val", "test"]:
    for c in ["NORMAL", "PNEUMONIA"]:
        print(s, c, len(list((out/s/c).glob("*"))))

## Train

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8x-cls.pt")   # alternatives: yolov8l-cls.pt, yolo11x-cls.pt

model.train(
    data="/content/cxr",
    epochs=30,
    imgsz=224,
    batch=32,
    patience=8,
    optimizer="AdamW",
    lr0=1e-3,
    fliplr=0.0,        # don't mirror X-rays
    degrees=10,
    translate=0.1,
    scale=0.2,
    project="/content/runs",
    name="pneumonia",
    workers=2,
)

## Recall-optimized thresholding
Missing a sick patient is worse than a false alarm, so pick the highest threshold on the **validation** set that reaches the target recall, then report on the untouched test set.

In [ ]:
import torch, torch.nn as nn, numpy as np
from PIL import Image
from sklearn.metrics import recall_score, precision_score, confusion_matrix, roc_auc_score

device = "cuda"
best = YOLO("/content/runs/pneumonia/weights/best.pt")
net = best.model.to(device).float().eval()
for p in net.parameters():
    p.requires_grad = True
print(best.names)   # expect {0: 'NORMAL', 1: 'PNEUMONIA'}

class Wrap(nn.Module):
    """Always returns logits as a single tensor."""
    def __init__(self, m):
        super().__init__()
        self.m = m
    def forward(self, x):
        o = self.m(x)
        return o[1] if isinstance(o, (tuple, list)) else o

wnet = Wrap(net).to(device).eval()

def load(f):
    im = Image.open(f).convert("RGB").resize((224, 224))
    return torch.from_numpy(np.asarray(im)).permute(2, 0, 1).float() / 255

def get_probs(split):
    paths, labels = [], []
    for i, c in enumerate(["NORMAL", "PNEUMONIA"]):
        fs = sorted((out/split/c).glob("*"))
        paths += fs; labels += [i]*len(fs)
    probs = []
    with torch.no_grad():
        for i in range(0, len(paths), 64):
            x = torch.stack([load(f) for f in paths[i:i+64]]).to(device)
            probs += torch.softmax(wnet(x), dim=1)[:, 1].cpu().tolist()
    return np.array(probs), np.array(labels), paths

pv, yv, _ = get_probs("val")

TARGET_RECALL = 0.98
best_t = 0.5
for t in np.arange(0.99, 0.01, -0.01):
    if recall_score(yv, pv >= t) >= TARGET_RECALL:
        best_t = float(t); break
print("Chosen threshold:", round(best_t, 2))   # use this value in the Streamlit sidebar

pt, yt, test_paths = get_probs("test")
pred = pt >= best_t
print("Test recall (sensitivity):", recall_score(yt, pred))
print("Test precision:", precision_score(yt, pred))
print("Test ROC-AUC:", roc_auc_score(yt, pt))
print("Confusion matrix [[TN FP],[FN TP]]:\n", confusion_matrix(yt, pred))

## Grad-CAM

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import matplotlib.pyplot as plt

cam = GradCAM(model=wnet, target_layers=[net.model[-2]])

def show_cam(idx_list):
    fig, axes = plt.subplots(2, len(idx_list), figsize=(4*len(idx_list), 8))
    for j, i in enumerate(idx_list):
        x = load(test_paths[i]).unsqueeze(0).to(device)
        heat = cam(input_tensor=x, targets=[ClassifierOutputTarget(1)])[0]
        rgb = x[0].permute(1, 2, 0).cpu().numpy()
        axes[0, j].imshow(rgb); axes[0, j].axis("off")
        axes[0, j].set_title(f"{'PNEUMONIA' if yt[i] else 'NORMAL'} (p={pt[i]:.2f})")
        axes[1, j].imshow(show_cam_on_image(rgb, heat, use_rgb=True)); axes[1, j].axis("off")
    plt.tight_layout(); plt.savefig("gradcam.png", dpi=150); plt.show()

pneu = np.where(yt == 1)[0][:3]
norm = np.where(yt == 0)[0][:3]
show_cam(list(pneu) + list(norm))

## Save and download

In [ ]:
import shutil
from google.colab import files, drive

shutil.copy("/content/runs/pneumonia/weights/best.pt", "best.pt")
files.download("best.pt")
files.download("gradcam.png")

drive.mount("/content/drive")
shutil.copy("best.pt", "/content/drive/MyDrive/best.pt")